In [17]:
# 환경 설정
import os
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

client = OpenAI()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print("✅ 환경 설정 완료")

✅ 환경 설정 완료


In [28]:
# ============================================================
# 주택청약 FAQ 샘플 데이터 (실습용)
# ============================================================
SAMPLE_FAQ_DATA = [
    {"id": "FAQ001", "category": "청약통장",
     "question": "주택청약종합저축이란 무엇인가요?",
     "answer": "주택청약종합저축은 국민주택과 민영주택 모두에 청약할 수 있는 만능 통장입니다.\n1) 매월 2만원~50만원 자유 납입\n2) 가입 후 일정 기간 경과 시 청약 자격 부여\n3) 2009년 5월 이후 모든 청약통장이 통합됨",
     "keywords": ["청약종합저축", "만능통장", "납입", "가입"], "difficulty": "easy"},
    {"id": "FAQ004", "category": "청약통장",
     "question": "청약통장 1순위 조건은 무엇인가요?",
     "answer": "1순위 조건은 주택 유형에 따라 다릅니다.\n1) 민영주택: 수도권 12개월, 비수도권 6개월 + 예치금\n2) 국민주택: 수도권 12개월(24회), 비수도권 6개월(12회)\n3) 투기과열지구: 2년, 24회 납입",
     "keywords": ["1순위", "가입기간", "예치금", "투기과열지구"], "difficulty": "medium"},
    {"id": "FAQ005", "category": "청약자격",
     "question": "주택 청약 신청 자격 조건은 무엇인가요?",
     "answer": "1) 만 19세 이상 (기혼자는 연령 제한 없음)\n2) 청약통장 가입 필수\n3) 국민주택: 무주택 세대구성원\n4) 민영주택: 세대주 또는 세대원 가능\n※ 투기과열지구는 세대주만 청약 가능",
     "keywords": ["청약자격", "만19세", "무주택", "세대주"], "difficulty": "easy"},
    {"id": "FAQ006", "category": "청약자격",
     "question": "무주택자 기준은 무엇인가요?",
     "answer": "본인과 세대원 모두 주택 미소유 시 무주택자입니다.\n예외: 60세 이상 직계존속 소유 주택, 20㎡ 이하 소형주택, 상속 후 3개월 내 처분 주택\n※ 분양권/입주권도 주택 수에 포함",
     "keywords": ["무주택", "세대원", "소형주택", "분양권"], "difficulty": "medium"},
    {"id": "FAQ009", "category": "특별공급",
     "question": "특별공급의 종류에는 어떤 것이 있나요?",
     "answer": "1) 기관추천 (국가유공자, 장애인 등)\n2) 다자녀가구 (3명 이상)\n3) 신혼부부 (혼인 7년 이내)\n4) 생애최초 (최초 주택 구입)\n5) 노부모부양 (만 65세 이상 부모)\n※ 2021년부터 신혼/생애최초 물량 확대",
     "keywords": ["특별공급", "기관추천", "다자녀", "신혼부부", "생애최초"], "difficulty": "medium"},
    {"id": "FAQ010", "category": "특별공급",
     "question": "신혼부부 특별공급 조건은 무엇인가요?",
     "answer": "1) 혼인기간 7년 이내 무주택 세대주\n2) 소득: 도시근로자 월평균소득 100~140%\n3) 전용면적 85㎡ 이하\n4) 혼인기간 짧을수록 + 자녀 많을수록 가점 높음\n5) 예비 신혼부부도 신청 가능",
     "keywords": ["신혼부부", "혼인기간", "소득기준", "가점"], "difficulty": "medium"},
    {"id": "FAQ013", "category": "일반공급",
     "question": "가점제와 추첨제의 차이는 무엇인가요?",
     "answer": "가점제: 무주택기간+부양가족+가입기간으로 점수화 (84점 만점)\n추첨제: 무작위 추첨\n1) 투기과열지구: 가점제 100%\n2) 청약과열지역: 가점 75% + 추첨 25%\n3) 기타: 가점 40% + 추첨 60%",
     "keywords": ["가점제", "추첨제", "84점", "투기과열지구"], "difficulty": "medium"},
    {"id": "FAQ017", "category": "당첨/계약",
     "question": "당첨자 발표는 어떻게 확인하나요?",
     "answer": "1) 청약홈(www.applyhome.co.kr) 접속\n2) 당첨자 조회 메뉴 클릭\n3) 문자 알림 서비스 신청 가능\n※ 당첨 후 서류 제출 기간과 계약 일정 반드시 확인",
     "keywords": ["당첨자발표", "청약홈", "SMS알림", "서류제출"], "difficulty": "easy"},
    {"id": "FAQ020", "category": "당첨/계약",
     "question": "재당첨 제한이란 무엇인가요?",
     "answer": "당첨 후 일정 기간 다른 주택 청약 불가:\n1) 투기과열지구: 10년\n2) 청약과열지역: 7년\n3) 수도권 공공주택: 5년\n※ 세대원 전원 적용 (배우자 당첨 시 본인도 제한)",
     "keywords": ["재당첨제한", "10년", "7년", "세대원"], "difficulty": "medium"},
    {"id": "FAQ023", "category": "기타",
     "question": "청약홈 사이트는 어떻게 이용하나요?",
     "answer": "청약홈(www.applyhome.co.kr) - 한국부동산원 운영\n1) 회원가입 후 공인인증서/간편인증 로그인\n2) 청약 신청, 당첨 확인, 가점 계산 가능\n3) 모바일 앱(청약홈)도 동일 서비스 제공",
     "keywords": ["청약홈", "공인인증서", "간편인증", "가점계산"], "difficulty": "easy"},
]

SAMPLE_TEST_QUERIES = [
    {"query": "청약통장 가입하려면 어떻게 해요?", "expected_category": "청약통장", "expected_faq_id": "FAQ001"},
    {"query": "1순위 되려면 뭐가 필요해요?", "expected_category": "청약통장", "expected_faq_id": "FAQ004"},
    {"query": "신혼부부 특공 자격이 궁금해요", "expected_category": "특별공급", "expected_faq_id": "FAQ010"},
    {"query": "가점이 높으면 유리한가요?", "expected_category": "일반공급", "expected_faq_id": "FAQ013"},
    {"query": "당첨되면 어떻게 확인해요?", "expected_category": "당첨/계약", "expected_faq_id": "FAQ017"},
]

print(f"📦 FAQ 데이터 로드 완료: {len(SAMPLE_FAQ_DATA)}개 QA, {len(SAMPLE_TEST_QUERIES)}개 테스트 질의")

📦 FAQ 데이터 로드 완료: 10개 QA, 5개 테스트 질의


In [54]:
# 사이클 1 : OpenAI API로 주택청약 관련 질문을 보내고 답변을 받아보세요. `system` 역할에 "주택청약 전문 상담원"을 설정하세요.
question = "청약통장 1순위가 되려면 어떤 조건이 필요하나요?"
response = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0,
    messages=[
        {"role": "system", "content": "당신은 주택청약 전문 상담원입니다. 질문에 친절하고 정확하게 답변해 주세요."},
        {"role": "user", "content": question},
    ],
)
answer = response.choices[0].message.content
print("질문:", question)
print("답변:", answer)

답변: 청약통장 1순위가 되기 위해서는 다음과 같은 조건을 충족해야 합니다:

1. **가입 기간**: 청약통장에 가입한 지 최소 24개월(2년)이 지나야 합니다. 다만, 주택청약종합저축의 경우 가입 후 12개월이 지나면 1순위로 인정받을 수 있는 경우도 있습니다.

2. **납입 횟수**: 청약통장에 일정 횟수 이상 납입해야 합니다. 일반적으로는 최소 12회 이상 납입해야 1순위로 인정됩니다.

3. **주택 소유 여부**: 본인 또는 세대원이 주택을 소유하고 있지 않아야 합니다. 주택 소유가 있는 경우 1순위 자격이 제한됩니다.

4. **청약 신청 지역**: 청약을 신청하는 지역에 따라 추가적인 조건이 있을 수 있습니다. 예를 들어, 해당 지역의 주택 수요와 공급에 따라 다를 수 있습니다.

이 외에도 특정 지역이나 주택 유형에 따라 추가적인 조건이 있을 수 있으니, 청약을 신청하기 전에 해당 조건을 잘 확인하는 것이 중요합니다.


In [56]:
# 사이클 2 : `SAMPLE_FAQ_DATA`에서 카테고리별 FAQ 개수를 세고, `difficulty`가 `"easy"`인 항목만 필터링해서 출력하세요.
# 1) 카테고리별 FAQ 개수
from collections import Counter

categories = [item["category"] for item in SAMPLE_FAQ_DATA]
category_counts = Counter(categories)
for cat, count in sorted(category_counts.items(), key=lambda x: -x[1]):
    print(f"  {cat}: {count}개")
print()

# 2) difficulty가 "easy"인 항목만 필터링
easy_faqs = [item for item in SAMPLE_FAQ_DATA if item["difficulty"] == "easy"]
for faq in easy_faqs:
    print(f"  [{faq['id']}] {faq['category']} - {faq['question']}")

  청약통장: 2개
  청약자격: 2개
  특별공급: 2개
  당첨/계약: 2개
  일반공급: 1개
  기타: 1개

  [FAQ001] 청약통장 - 주택청약종합저축이란 무엇인가요?
  [FAQ005] 청약자격 - 주택 청약 신청 자격 조건은 무엇인가요?
  [FAQ017] 당첨/계약 - 당첨자 발표는 어떻게 확인하나요?
  [FAQ023] 기타 - 청약홈 사이트는 어떻게 이용하나요?


In [57]:
# 사이클 3: FAQ 검색 함수
def search_faq(query: str, faq_data: list, top_k: int = 3) -> list:
    q = query.strip()
    scored = []
    for item in faq_data:
        keywords = item.get("keywords", [])
        score = sum(1 for kw in keywords if kw in q)
        scored.append((score, item))
    scored.sort(key=lambda x: -x[0])
    return [item for _, item in scored[:top_k]]


# SAMPLE_TEST_QUERIES로 테스트
for t in SAMPLE_TEST_QUERIES:
    q = t["query"]
    results = search_faq(q, SAMPLE_FAQ_DATA)
    print(f"질의: {q}")
    print(f"기대 FAQ ID: {t['expected_faq_id']}")
    print(f"검색 결과: {[r['id'] for r in results]}")
    for r in results:
        print(f"    - [{r['id']}] {r['question']}")

질의: 청약통장 가입하려면 어떻게 해요?
기대 FAQ ID: FAQ001
검색 결과: ['FAQ001', 'FAQ004', 'FAQ005']
    - [FAQ001] 주택청약종합저축이란 무엇인가요?
    - [FAQ004] 청약통장 1순위 조건은 무엇인가요?
    - [FAQ005] 주택 청약 신청 자격 조건은 무엇인가요?
질의: 1순위 되려면 뭐가 필요해요?
기대 FAQ ID: FAQ004
검색 결과: ['FAQ004', 'FAQ001', 'FAQ005']
    - [FAQ004] 청약통장 1순위 조건은 무엇인가요?
    - [FAQ001] 주택청약종합저축이란 무엇인가요?
    - [FAQ005] 주택 청약 신청 자격 조건은 무엇인가요?
질의: 신혼부부 특공 자격이 궁금해요
기대 FAQ ID: FAQ010
검색 결과: ['FAQ009', 'FAQ010', 'FAQ001']
    - [FAQ009] 특별공급의 종류에는 어떤 것이 있나요?
    - [FAQ010] 신혼부부 특별공급 조건은 무엇인가요?
    - [FAQ001] 주택청약종합저축이란 무엇인가요?
질의: 가점이 높으면 유리한가요?
기대 FAQ ID: FAQ013
검색 결과: ['FAQ010', 'FAQ001', 'FAQ004']
    - [FAQ010] 신혼부부 특별공급 조건은 무엇인가요?
    - [FAQ001] 주택청약종합저축이란 무엇인가요?
    - [FAQ004] 청약통장 1순위 조건은 무엇인가요?
질의: 당첨되면 어떻게 확인해요?
기대 FAQ ID: FAQ017
검색 결과: ['FAQ001', 'FAQ004', 'FAQ005']
    - [FAQ001] 주택청약종합저축이란 무엇인가요?
    - [FAQ004] 청약통장 1순위 조건은 무엇인가요?
    - [FAQ005] 주택 청약 신청 자격 조건은 무엇인가요?


In [60]:
# 사이클 4: 검색 결과 + LLM 답변 생성
def ask_faq(question: str, faq_data: list, llm) -> dict:
    """
    질문으로 FAQ 검색 → 검색된 FAQ를 system에 넣어 LLM 답변 생성.
    반환: {"answer": str, "referenced_faqs": list}
    """
    # 1) 키워드 검색으로 관련 FAQ top 3
    refs = search_faq(question, faq_data)

    # 2) 검색된 FAQ를 context 문자열로
    context_parts = []
    for i, faq in enumerate(refs, 1):
        context_parts.append(
            f"[{i}] (ID: {faq['id']}, 카테고리: {faq['category']})\n"
            f"Q: {faq['question']}\nA: {faq['answer']}"
        )
    context = "\n\n".join(context_parts)

    # 3) LangChain: prompt | llm | parser
    prompt = ChatPromptTemplate.from_messages([
        ("system", "당신은 주택청약 전문 상담원입니다. 아래 FAQ를 참고하여 질문에만 답변하세요. "
         "참고 FAQ를 우선 반영하세요.\n\n=== 참고 FAQ ===\n{context}"),
        ("human", "{question}"),
    ])
    chain = prompt | llm | StrOutputParser()
    answer = chain.invoke({"context": context, "question": question})

    # 4) 참고 FAQ 목록
    referenced_faqs = [{"id": r["id"], "category": r["category"], "question": r["question"]} for r in refs]

    return {"answer": answer, "referenced_faqs": referenced_faqs}


# 테스트
test_question = SAMPLE_TEST_QUERIES[0]["query"]
result = ask_faq(test_question, SAMPLE_FAQ_DATA, llm)
print("질문:", test_question)
print("\n답변:", result["answer"])
print("\n참고 FAQ:", [f"{f['id']} - {f['question']}" for f in result["referenced_faqs"]])

질문: 청약통장 가입하려면 어떻게 해요?

답변: 청약통장 가입은 다음과 같은 절차를 따릅니다:

1. **은행 선택**: 주택청약종합저축을 취급하는 은행을 선택합니다. 대부분의 주요 은행에서 가입할 수 있습니다.
2. **신청서 작성**: 해당 은행의 청약통장 신청서를 작성합니다.
3. **신분증 제출**: 본인 확인을 위해 신분증(주민등록증, 운전면허증 등)을 제출합니다.
4. **초기 납입**: 최소한의 초기 납입금을 입금합니다. (은행에 따라 다를 수 있습니다)
5. **가입 완료**: 모든 절차가 완료되면 청약통장이 개설됩니다.

가입 후에는 매월 2만원에서 50만원까지 자유롭게 납입할 수 있습니다.

참고 FAQ: ['FAQ001 - 주택청약종합저축이란 무엇인가요?', 'FAQ004 - 청약통장 1순위 조건은 무엇인가요?', 'FAQ005 - 주택 청약 신청 자격 조건은 무엇인가요?']


In [61]:
# 사이클 5 PromptTemplate
# 1) FAQ 답변용: {context}, {question}
faq_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 주택청약 전문 상담원입니다. 아래 참고 자료를 바탕으로 질문에 답변하세요.\n\n참고 자료:\n{context}"),
    ("human", "{question}"),
])

# 2) 카테고리 분류용: {question} → 카테고리 하나 반환
categories = "청약통장, 청약자격, 특별공급, 일반공급, 당첨/계약, 기타"
category_prompt = ChatPromptTemplate.from_messages([
    ("system", f"주택청약 FAQ 카테고리 분류자입니다. 주어진 질문을 다음 카테고리 중 하나로만 분류하세요: {categories}. 카테고리 이름만 출력하세요."),
    ("human", "{question}"),
])

# --- FAQ 답변용 테스트 ---
# 검색된 FAQ를 context 문자열로 만들고 question 넣기
question = "청약통장 1순위 조건이 뭔가요?"
sample_refs = search_faq(question, SAMPLE_FAQ_DATA, top_k=2)
context_str = "\n\n".join(
    f"Q: {r['question']}\nA: {r['answer']}" for r in sample_refs
)
faq_messages = faq_prompt.invoke({"context": context_str, "question": question})
print("FAQ 답변용 프롬프트 (messages):", faq_messages)
# 실제 답변까지 받으려면:
faq_chain = faq_prompt | llm | StrOutputParser()
answer = faq_chain.invoke({"context": context_str, "question": question})
print("답변:", answer[:200], "..." if len(answer) > 200 else "")

# --- 카테고리 분류용 테스트 ---
category_chain = category_prompt | llm | StrOutputParser()
for t in SAMPLE_TEST_QUERIES[:3]:
    q = t["query"]
    out = category_chain.invoke({"question": q})
    print(f"질의: {q}")
    print(f"  기대 카테고리: {t['expected_category']} → 분류 결과: {out.strip()}\n")

FAQ 답변용 프롬프트 (messages): messages=[SystemMessage(content='당신은 주택청약 전문 상담원입니다. 아래 참고 자료를 바탕으로 질문에 답변하세요.\n\n참고 자료:\nQ: 청약통장 1순위 조건은 무엇인가요?\nA: 1순위 조건은 주택 유형에 따라 다릅니다.\n1) 민영주택: 수도권 12개월, 비수도권 6개월 + 예치금\n2) 국민주택: 수도권 12개월(24회), 비수도권 6개월(12회)\n3) 투기과열지구: 2년, 24회 납입\n\nQ: 주택청약종합저축이란 무엇인가요?\nA: 주택청약종합저축은 국민주택과 민영주택 모두에 청약할 수 있는 만능 통장입니다.\n1) 매월 2만원~50만원 자유 납입\n2) 가입 후 일정 기간 경과 시 청약 자격 부여\n3) 2009년 5월 이후 모든 청약통장이 통합됨', additional_kwargs={}, response_metadata={}), HumanMessage(content='청약통장 1순위 조건이 뭔가요?', additional_kwargs={}, response_metadata={})]
답변: 청약통장 1순위 조건은 주택 유형에 따라 다릅니다.

1) **민영주택**: 수도권에서 12개월 이상, 비수도권에서 6개월 이상 예치금이 필요합니다.
2) **국민주택**: 수도권에서 12개월(24회) 이상, 비수도권에서 6개월(12회) 이상 예치금이 필요합니다.
3) **투기과열지구**: 2년 이상, 24회 납입이 필요합니다.

각 조건을 충족해야 1순위 ...
질의: 청약통장 가입하려면 어떻게 해요?
  기대 카테고리: 청약통장 → 분류 결과: 청약통장

질의: 1순위 되려면 뭐가 필요해요?
  기대 카테고리: 청약통장 → 분류 결과: 청약자격

질의: 신혼부부 특공 자격이 궁금해요
  기대 카테고리: 특별공급 → 분류 결과: 특별공급



In [62]:
# 사이클 6 : LCEL 체인
# FAQ 답변용 프롬프트 (context + question)
prompt = ChatPromptTemplate.from_messages([
    ("system", "주택청약 전문 상담원입니다. 아래 참고 자료를 바탕으로 답변하세요.\n\n{context}"),
    ("human", "{question}"),
])
# 체인: prompt → llm → 문자열
faq_chain = prompt | llm | StrOutputParser()
# 질문 2개 테스트
questions = ["청약통장 1순위 조건이 뭔가요?", "신혼부부 특별공급 조건이 궁금해요"]
for q in questions:
    ctx = "\n".join(f"Q: {r['question']}\nA: {r['answer']}" for r in search_faq(q, SAMPLE_FAQ_DATA, top_k=2))
    print("질문:", q)
    print("답변:", faq_chain.invoke({"context": ctx, "question": q}))
    print()
# 스트리밍 한 번 해보기
question = "당첨자 어떻게 확인해요?"
ctx = "\n".join(f"Q: {r['question']}\nA: {r['answer']}" for r in search_faq(question, SAMPLE_FAQ_DATA, top_k=2))
print("스트리밍:")
for chunk in faq_chain.stream({"context": ctx, "question": question}):
    print(chunk, end="")

질문: 청약통장 1순위 조건이 뭔가요?
답변: 청약통장 1순위 조건은 주택 유형에 따라 다릅니다.

1) **민영주택**: 수도권에서 12개월 이상, 비수도권에서 6개월 이상 예치금이 필요합니다.
2) **국민주택**: 수도권에서 12개월(24회) 이상, 비수도권에서 6개월(12회) 이상 예치금이 필요합니다.
3) **투기과열지구**: 2년 이상, 24회 납입이 필요합니다.

각 조건을 충족해야 1순위로 청약할 수 있습니다.

질문: 신혼부부 특별공급 조건이 궁금해요
답변: 신혼부부 특별공급의 조건은 다음과 같습니다:

1. **혼인기간**: 혼인기간이 7년 이내의 무주택 세대주여야 합니다.
2. **소득**: 도시근로자 월평균소득의 100%에서 140% 이내여야 합니다.
3. **전용면적**: 신청하는 주택의 전용면적은 85㎡ 이하이어야 합니다.
4. **가점제**: 혼인기간이 짧을수록, 자녀가 많을수록 가점이 높아집니다.
5. **예비 신혼부부**: 예비 신혼부부도 신청이 가능합니다.

이 조건들을 충족하면 신혼부부 특별공급에 신청할 수 있습니다. 추가적인 질문이 있으시면 언제든지 문의해 주세요!

스트리밍:
주택청약 당첨자는 다음과 같은 방법으로 확인할 수 있습니다:

1. **청약홈**: 청약홈 웹사이트에 접속하여 본인의 청약 신청 내역을 확인할 수 있습니다. 로그인 후 '청약결과' 메뉴에서 확인 가능합니다.

2. **문자 알림**: 청약 신청 시 등록한 휴대폰 번호로 당첨 결과에 대한 문자 알림을 받을 수 있습니다.

3. **주택청약 관련 기관**: 해당 주택의 공급 기관(예: 한국토지주택공사, 지방자치단체 등)에서 발표하는 당첨자 명단을 확인할 수 있습니다.

당첨 결과는 보통 청약 접수 마감 후 일정 기간 내에 발표되므로, 해당 기간을 확인하고 방법에 따라 확인하시면 됩니다.

In [33]:
# 사이클 7 : 검색
# 질문만 넣으면 답변 반환 (FAQ 검색 없음)
prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 주택청약 전문 상담원입니다. 질문에 친절하고 정확하게 답변해 주세요."),
    ("human", "{question}"),
])
chain = prompt | llm | StrOutputParser()

# 테스트 5개
for t in SAMPLE_TEST_QUERIES:
    q = t["query"]
    answer = chain.invoke({"question": q})
    print(f"Q: {q}\nA: {answer}\n")

Q: 청약통장 가입하려면 어떻게 해요?
A: 청약통장에 가입하려면 다음과 같은 절차를 따르면 됩니다:

1. **은행 선택**: 청약통장은 주택청약을 취급하는 은행에서 가입할 수 있습니다. 주요 은행으로는 국민은행, 신한은행, 우리은행, 농협 등이 있습니다.

2. **신분증 준비**: 가입 시 본인 확인을 위해 신분증(주민등록증, 운전면허증 등)을 준비해야 합니다.

3. **가입 신청**: 선택한 은행의 지점에 방문하여 청약통장 가입을 신청합니다. 은행 직원에게 청약통장 가입을 원한다고 말씀하시면 됩니다.

4. **가입 조건 확인**: 청약통장은 일반청약과 신혼부부청약 등 여러 종류가 있으므로, 본인의 상황에 맞는 통장을 선택해야 합니다. 각 통장마다 가입 조건과 혜택이 다를 수 있습니다.

5. **가입금액 결정**: 청약통장에 가입할 때 초기 납입금액을 결정해야 합니다. 최소 가입금액은 은행마다 다를 수 있으니 확인이 필요합니다.

6. **계약서 작성 및 서명**: 가입 신청서와 관련 서류를 작성하고 서명합니다.

7. **통장 발급**: 모든 절차가 완료되면 청약통장이 발급됩니다.

가입 후에는 정기적으로 납입금을 입금하여 청약 자격을 유지해야 하며, 청약 신청 시 필요한 조건을 충족해야 합니다. 추가적인 질문이 있으시면 언제든지 문의해 주세요!

Q: 1순위 되려면 뭐가 필요해요?
A: 주택청약에서 1순위를 받기 위해서는 다음과 같은 조건을 충족해야 합니다:

1. **청약통장 가입 기간**: 청약통장에 가입한 지 최소 1년 이상이어야 합니다. 
2. **납입 횟수**: 청약통장에 일정 횟수 이상 납입해야 합니다. 일반적으로는 12회 이상 납입해야 1순위 자격을 갖출 수 있습니다.
3. **주택 소유 여부**: 본인 또는 세대원이 주택을 소유하고 있지 않아야 합니다. 주택 소유가 있는 경우, 일정 기간(보통 5년) 동안 청약을 신청할 수 없습니다.
4. **소득 기준**: 일부 주택청약 상품은 소득 기준이 있을 수 있으므로, 해당 조건도 확인해야

In [63]:
# 사이클 8 : 에러 처리
def safe_ask(question: str):
    q = (question or "").strip()
    if not q:
        return "질문을 입력해 주세요."
    if len(q) > 500:
        return "질문은 500자 이내로 입력해 주세요."
    if q.isdigit():
        return "숫자만으로는 답변할 수 없습니다."
    try:
        return chain.invoke({"question": q})
    except Exception as e:
        return f"오류가 발생했습니다: {e}"

# 테스트 6가지
for q in ["", "   ", "12345", "x" * 501, "청약통장 1순위 조건이 뭔가요?", "신혼부부 특공 자격이 궁금해요"]:
    print(f"입력: {repr(q)[:35]}\n→ {safe_ask(q)[:80]}...\n")

입력: ''
→ 질문을 입력해 주세요....

입력: '   '
→ 질문을 입력해 주세요....

입력: '12345'
→ 숫자만으로는 답변할 수 없습니다....

입력: 'xxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxxx
→ 질문은 500자 이내로 입력해 주세요....

입력: '청약통장 1순위 조건이 뭔가요?'
→ 청약통장 1순위 조건은 다음과 같습니다:

1. **가입 기간**: 청약통장에 가입한 지 최소 2년이 지나야 합니다. 다만, 주택청약종합저축의 ...

입력: '신혼부부 특공 자격이 궁금해요'
→ 신혼부부 특별공급(특공)은 주택청약 제도 중 하나로, 신혼부부가 주택을 구매할 때 우선적으로 공급받을 수 있는 제도입니다. 신혼부부 특공의 자격...



In [2]:
# 사이클 9 : Gradio 채팅 UI
import gradio as gr

def chat_fn(message, history):
    return safe_ask(message)

gr.ChatInterface(
    chat_fn,
    title="주택청약 FAQ 챗봇",
    description="주택청약 관련 질문을 입력하세요.",
    examples=[
        "청약통장 가입하려면 어떻게 해요?",
        "1순위 되려면 뭐가 필요해요?",
        "신혼부부 특공 자격이 궁금해요",
        "가점이 높으면 유리한가요?",
        "당첨되면 어떻게 확인해요?",
    ],
).launch()

* Running on local URL:  http://127.0.0.1:7871
* To create a public link, set `share=True` in `launch()`.


In [3]:
# 사이클 10: 최종 통합 테스트
import time
import gradio as gr
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 실습용 FAQ 데이터
SAMPLE_FAQ_DATA = [
    {"id": "FAQ001", "category": "청약통장", "question": "주택청약종합저축이란 무엇인가요?", "answer": "주택청약종합저축은 국민주택과 민영주택 모두에 청약할 수 있는 만능 통장입니다.\n1) 매월 2만원~50만원 자유 납입\n2) 가입 후 일정 기간 경과 시 청약 자격 부여\n3) 2009년 5월 이후 모든 청약통장이 통합됨", "keywords": ["청약종합저축", "만능통장", "납입", "가입"], "difficulty": "easy"},
    {"id": "FAQ004", "category": "청약통장", "question": "청약통장 1순위 조건은 무엇인가요?", "answer": "1순위 조건은 주택 유형에 따라 다릅니다.\n1) 민영주택: 수도권 12개월, 비수도권 6개월 + 예치금\n2) 국민주택: 수도권 12개월(24회), 비수도권 6개월(12회)\n3) 투기과열지구: 2년, 24회 납입", "keywords": ["1순위", "가입기간", "예치금", "투기과열지구"], "difficulty": "medium"},
    {"id": "FAQ005", "category": "청약자격", "question": "주택 청약 신청 자격 조건은 무엇인가요?", "answer": "1) 만 19세 이상 (기혼자는 연령 제한 없음)\n2) 청약통장 가입 필수\n3) 국민주택: 무주택 세대구성원\n4) 민영주택: 세대주 또는 세대원 가능\n※ 투기과열지구는 세대주만 청약 가능", "keywords": ["청약자격", "만19세", "무주택", "세대주"], "difficulty": "easy"},
    {"id": "FAQ006", "category": "청약자격", "question": "무주택자 기준은 무엇인가요?", "answer": "본인과 세대원 모두 주택 미소유 시 무주택자입니다.\n예외: 60세 이상 직계존속 소유 주택, 20㎡ 이하 소형주택, 상속 후 3개월 내 처분 주택\n※ 분양권/입주권도 주택 수에 포함", "keywords": ["무주택", "세대원", "소형주택", "분양권"], "difficulty": "medium"},
    {"id": "FAQ009", "category": "특별공급", "question": "특별공급의 종류에는 어떤 것이 있나요?", "answer": "1) 기관추천 (국가유공자, 장애인 등)\n2) 다자녀가구 (3명 이상)\n3) 신혼부부 (혼인 7년 이내)\n4) 생애최초 (최초 주택 구입)\n5) 노부모부양 (만 65세 이상 부모)\n※ 2021년부터 신혼/생애최초 물량 확대", "keywords": ["특별공급", "기관추천", "다자녀", "신혼부부", "생애최초"], "difficulty": "medium"},
    {"id": "FAQ010", "category": "특별공급", "question": "신혼부부 특별공급 조건은 무엇인가요?", "answer": "1) 혼인기간 7년 이내 무주택 세대주\n2) 소득: 도시근로자 월평균소득 100~140%\n3) 전용면적 85㎡ 이하\n4) 혼인기간 짧을수록 + 자녀 많을수록 가점 높음\n5) 예비 신혼부부도 신청 가능", "keywords": ["신혼부부", "혼인기간", "소득기준", "가점"], "difficulty": "medium"},
    {"id": "FAQ013", "category": "일반공급", "question": "가점제와 추첨제의 차이는 무엇인가요?", "answer": "가점제: 무주택기간+부양가족+가입기간으로 점수화 (84점 만점)\n추첨제: 무작위 추첨\n1) 투기과열지구: 가점제 100%\n2) 청약과열지역: 가점 75% + 추첨 25%\n3) 기타: 가점 40% + 추첨 60%", "keywords": ["가점제", "추첨제", "84점", "투기과열지구"], "difficulty": "medium"},
    {"id": "FAQ017", "category": "당첨/계약", "question": "당첨자 발표는 어떻게 확인하나요?", "answer": "1) 청약홈(www.applyhome.co.kr) 접속\n2) 당첨자 조회 메뉴 클릭\n3) 문자 알림 서비스 신청 가능\n※ 당첨 후 서류 제출 기간과 계약 일정 반드시 확인", "keywords": ["당첨자발표", "청약홈", "SMS알림", "서류제출"], "difficulty": "easy"},
    {"id": "FAQ020", "category": "당첨/계약", "question": "재당첨 제한이란 무엇인가요?", "answer": "당첨 후 일정 기간 다른 주택 청약 불가:\n1) 투기과열지구: 10년\n2) 청약과열지역: 7년\n3) 수도권 공공주택: 5년\n※ 세대원 전원 적용 (배우자 당첨 시 본인도 제한)", "keywords": ["재당첨제한", "10년", "7년", "세대원"], "difficulty": "medium"},
    {"id": "FAQ023", "category": "기타", "question": "청약홈 사이트는 어떻게 이용하나요?", "answer": "청약홈(www.applyhome.co.kr) - 한국부동산원 운영\n1) 회원가입 후 공인인증서/간편인증 로그인\n2) 청약 신청, 당첨 확인, 가점 계산 가능\n3) 모바일 앱(청약홈)도 동일 서비스 제공", "keywords": ["청약홈", "공인인증서", "간편인증", "가점계산"], "difficulty": "easy"},
]

SAMPLE_TEST_QUERIES = [
    {"query": "청약통장 가입하려면 어떻게 해요?"},
    {"query": "1순위 되려면 뭐가 필요해요?"},
    {"query": "신혼부부 특공 자격이 궁금해요"},
    {"query": "가점이 높으면 유리한가요?"},
    {"query": "당첨되면 어떻게 확인해요?"},
]

def search_faq(query: str, faq_data: list, top_k: int = 3) -> list:
    q = (query or "").strip()
    scored = [
        (sum(1 for kw in item.get("keywords", []) if kw in q), item)
        for item in faq_data
    ]
    results = [item for score, item in sorted(scored, key=lambda x: -x[0]) if score > 0][:top_k]
    return results or faq_data[:1]


def format_chat_history(history, max_turns=10):
    if not history:
        return "(없음)"
    lines = []
    for item in history[-max_turns:]:
        if isinstance(item, dict):
            role = item.get("role")
            content = str(item.get("content") or "").strip()
            if role == "user":
                lines.append(f"사용자: {content}")
            elif role == "assistant":
                lines.append(f"상담원: {content}")
        elif isinstance(item, (list, tuple)) and len(item) >= 2:
            lines.append(f"사용자: {str(item[0]).strip()}")
            lines.append(f"상담원: {str(item[1]).strip()}")
    return "\n".join(lines) if lines else "(없음)"


def make_context(refs):
    return "\n\n".join(f"[FAQ ID: {faq['id']}]\n질문: {faq['question']}\n답변: {faq['answer']}" for faq in refs)


# FAQ 우선 답변, 없으면 AI가 주택청약 전문가로서 일반 지식으로 답변
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 주택청약 전문 상담원입니다. [참고 FAQ]에 관련 내용이 있으면 그 내용을 우선 활용해 답변하세요. 참고 FAQ에 해당 내용이 없거나 부족하면, 주택청약 전문가로서 일반 지식으로 답변해 주세요. 가능하면 참고한 FAQ ID를 언급하세요.\n\n[이전 대화]\n{chat_history}\n\n[참고 FAQ]\n{context}"),
    ("human", "{question}"),
])
chat_chain = chat_prompt | llm | StrOutputParser()


def build_inputs(question, history=None, faq_data=SAMPLE_FAQ_DATA, top_k=3):
    q = (question or "").strip()
    if not q:
        return None, "질문을 입력해 주세요.", []
    if len(q) > 500:
        return None, "질문은 500자 이내로 입력해 주세요.", []
    if q.isdigit():
        return None, "숫자만으로는 답변할 수 없습니다.", []
    refs = search_faq(q, faq_data, top_k)
    return {
        "chat_history": format_chat_history(history or []),
        "context": make_context(refs),
        "question": q,
    }, None, refs


def run_pipeline(question: str, history=None, faq_data=SAMPLE_FAQ_DATA, top_k=3):
    inputs, error_message, refs = build_inputs(question, history, faq_data, top_k)
    if error_message:
        return error_message, 0, 0.0
    try:
        start = time.time()
        answer = chat_chain.invoke(inputs)
        return answer, len(refs), round(time.time() - start, 2)
    except Exception as e:
        return f"오류가 발생했습니다: {e}", 0, 0.0

# 10개 질문 테스트
extra_questions = [
    "무주택자 기준이 뭔가요?",
    "재당첨 제한이 뭐예요?",
    "청약홈 어떻게 써요?",
    "가점제 추첨제 차이?",
    "특별공급 종류 알려줘요.",
]
test_questions = [item["query"] for item in SAMPLE_TEST_QUERIES] + extra_questions

print("질문\t\t\t\t참고FAQ수\t응답시간(초)\t답변요약")
print("-" * 75)
for question in test_questions:
    answer, ref_count, elapsed = run_pipeline(question)
    print(f"{question[:28]}\t{ref_count}\t{elapsed}\t{answer[:40]}...")

# Gradio UI: (1) 스트리밍 답변 (2) 이전 대화(클릭 후 사용자 메시지~AI 답변) 기억하여 채팅 이어짐
def ui_chat(message, history):
    inputs, error_message, _ = build_inputs(message, history, SAMPLE_FAQ_DATA, 3)
    if error_message:
        yield error_message
        return
    partial = ""
    for chunk in chat_chain.stream(inputs):
        partial += chunk
        yield partial  # 스트리밍: 한 토큰씩 누적하여 표시

gr.ChatInterface(
    fn=ui_chat,
    title="주택청약 FAQ 챗봇",
    description="FAQ 기반 답변 + 모르면 AI가 답변. 답변은 스트리밍으로 표시되며, 이전 대화를 기억해 채팅이 이어집니다.",
    examples=[
        "청약통장 가입하려면 어떻게 해요?",
        "1순위 되려면 뭐가 필요해요?",
        "신혼부부 특공 자격이 궁금해요",
        "가점이 높으면 유리한가요?",
        "당첨되면 어떻게 확인해요?",
    ],
).launch()

질문				참고FAQ수	응답시간(초)	답변요약
---------------------------------------------------------------------------
청약통장 가입하려면 어떻게 해요?	1	6.21	청약통장에 가입하려면 다음과 같은 절차를 따르면 됩니다:

1. **은행...
1순위 되려면 뭐가 필요해요?	1	3.16	1순위 조건은 주택 유형에 따라 다릅니다. 아래와 같이 정리할 수 있습니...
신혼부부 특공 자격이 궁금해요	2	2.9	신혼부부 특별공급의 자격 조건은 다음과 같습니다:

1. 혼인기간이 7년...
가점이 높으면 유리한가요?	1	2.78	네, 가점이 높으면 주택청약에서 유리합니다. 가점제는 신청자의 조건에 따...
당첨되면 어떻게 확인해요?	1	3.82	주택청약에 당첨되었는지 확인하는 방법은 다음과 같습니다:

1. **청약...
무주택자 기준이 뭔가요?	2	2.17	무주택자 기준은 본인과 세대원 모두 주택을 소유하지 않을 때 무주택자로 ...
재당첨 제한이 뭐예요?	1	3.67	재당첨 제한은 주택청약에 있어 동일한 주택 유형에 대해 일정 기간 내에 ...
청약홈 어떻게 써요?	2	3.13	청약홈을 이용하는 방법은 다음과 같습니다:

1. **회원가입**: 청약...
가점제 추첨제 차이?	2	3.08	가점제와 추첨제의 차이는 다음과 같습니다:

- **가점제**: 무주택기...
특별공급 종류 알려줘요.	1	3.01	특별공급의 종류는 다음과 같습니다:

1) 기관추천 (국가유공자, 장애인...
* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
